# Relax structure with MACE — Machine Learned Interatomic Potential

Use Zur and McGill superlattices matching [algorithm](https://doi.org/10.1063/1.3330840) to create interfaces between two materials using the `mat3ra-made` [implementation](https://github.com/Exabyte-io/made).

<h2 style="color:green">Usage</h2>

1. Drop the materials files into the "uploads" folder in the JupyterLab file browser
1. Set Input Parameters below or use the default values
1. Click "Run" > "Run All" to run all cells
1. Wait for the run to complete. Scroll down to view cell results.
1. Review the strain plot and modify parameters as needed

## Methodology

1. Load materials from JSON files and create substrate and film slabs via `mat3ra-made`
2. Run ZSL strain matching and plot strain vs interface area
3. Create the selected interface and convert to ASE atoms with `to_ase()`
4. Relax the interface with MACE-MP-0 and visualize convergence
5. Compute interface binding energy decomposition

## 1. Set Input Parameters
### 1.1. Interface and Relaxation Parameters


In [ ]:


FOLDER = "uploads"
INTERFACE_NAME = "Interface"  # Name of the interface to load from local file

RELAXATION_PARAMETERS = {
    "FMAX": 0.05,
}
MACE_MODEL = "large"  # "small", "medium", or "large" MACE-MP-0 foundation model

## 2. Install Packages

In [15]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|api_examples|torch|mace")

from mat3ra.notebooks_utils.pyodide.packages.torch import apply_all_patches

apply_all_patches()

To install packages, run `pip install ".[all]"` in the terminal
✓ Torch linalg patches applied
✓ Torch testing patches applied
✓ Matscipy patches applied
✓ LMDB and HDF5 stubs applied
✓ MACE tools patches applied

✅ All Pyodide patches applied successfully!


## 3. Load Materials

In [16]:
from mat3ra.made.material import Material
from mat3ra.notebooks_utils.material import load_material_from_folder
from mat3ra.standata.materials import Materials

interface = load_material_from_folder(FOLDER, INTERFACE_NAME) or Material.create(
    Materials.get_by_name_first_match(INTERFACE_NAME))

INFO: Found: 'C(001)-Ni(111), Interface'


### 3.1. Visualize Input Materials

In [17]:
from mat3ra.notebooks_utils.ipython.entity.material.visualize import ViewersEnum, visualize_materials as visualize

visualize([{"material": interface, "title": interface.name}], viewer=ViewersEnum.wave)
visualize(interface, repetitions=[1, 1, 1], rotation="-90x")

<IPython.core.display.Javascript object>

GridBox(children=(VBox(children=(Label(value='C2Ni3 - Material - rotation: -90x', layout=Layout(align_self='ce…

## 4. Apply Relaxation
### 4.1. Relax with MACE

In [ ]:
import plotly.graph_objs as go
from IPython.display import display
from plotly.subplots import make_subplots

from mat3ra.notebooks_utils.primitive.environment import is_pyodide_environment
from mat3ra.notebooks_utils.pyodide.packages.mace import get_mace_model_pyodide

from mat3ra.made.tools.convert import to_ase
from ase.optimize import BFGS
from mace.calculators import mace_mp

mace_mp = mace_mp if not is_pyodide_environment() else get_mace_model_pyodide
calculator = mace_mp(model=MACE_MODEL, dispersion=True, default_dtype="float32", device="cpu")

ase_interface = to_ase(interface)
ase_interface.set_calculator(calculator)
dyn = BFGS(ase_interface)

steps = []
energies = []

fig = make_subplots(rows=1, cols=1, specs=[[{"type": "scatter"}]])
scatter = go.Scatter(x=[], y=[], mode="lines+markers", name="Energy")
fig.add_trace(scatter)
fig.update_layout(title_text="Real-time Optimization Progress", xaxis_title="Step", yaxis_title="Energy (eV)")

f = go.FigureWidget(fig)
display(f)


def plotly_callback():
    step = dyn.nsteps
    energy = ase_interface.get_total_energy()
    steps.append(step)
    energies.append(energy)
    print(f"Step: {step}, Energy: {energy:.4f} eV")
    with f.batch_update():
        f.data[0].x = steps
        f.data[0].y = energies


dyn.attach(plotly_callback, interval=1)
dyn.run(fmax=RELAXATION_PARAMETERS["FMAX"])

ase_original_interface = to_ase(interface)
ase_original_interface.set_calculator(calculator)
ase_final_interface = ase_interface

original_energy = ase_original_interface.get_total_energy()
relaxed_energy = ase_interface.get_total_energy()

print(f"The final energy is {float(relaxed_energy):.3f} eV.")

## 5. Analyze Results
### 5.1. View Structure Before and After Relaxation

In [19]:
from mat3ra.made.tools.convert import from_ase

material_original = Material.create(from_ase(ase_original_interface))
material_relaxed = Material.create(from_ase(ase_final_interface))
material_original.name = interface.name
material_relaxed.name = interface.name + " (MACE Relaxed)"

visualize(
    [
        {"material": material_original, "title": material_original.name},
        {"material": material_relaxed, "title": material_relaxed.name},
    ],
    viewer=ViewersEnum.wave,
)

GridBox(children=(HTML(value='\n    <h2>C(001)-Ni(111), Interface</h2>\n    <div id="wave-0-1779062603.775706"…

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### 5.2. Output interlayer distance before and after relaxation
This requires labels for substrate and film present in the interface structure, which is already done if interface was created with `mat3ra-made`.

In [20]:
from mat3ra.made.tools.analyze.other import get_average_interlayer_distance

SUBSTRATE_TAG = 0
FILM_TAG = 1

print(
    f"Interlayer distance before relaxation: {get_average_interlayer_distance(material_original, SUBSTRATE_TAG, FILM_TAG):.4f} Å")
print(
    f"Interlayer distance after relaxation:  {get_average_interlayer_distance(material_relaxed, SUBSTRATE_TAG, FILM_TAG):.4f} Å")

Interlayer distance before relaxation: 3.0000 Å
Interlayer distance after relaxation:  4.2888 Å


### 5.3. Calculate Interface Energy
Interface should have labels marking substrate and film atoms with different tags (e.g. 0 for substrate and 1 for film) for the separation.

In [21]:
def filter_atoms_by_tag(atoms, material_index):
    return atoms[atoms.get_tags() == material_index]


def calculate_energy(atoms, calc):
    atoms.set_calculator(calc)
    return atoms.get_total_energy()


def calculate_delta_energy(total_energy, *component_energies):
    return total_energy - sum(component_energies)


substrate_original = filter_atoms_by_tag(ase_original_interface, SUBSTRATE_TAG)
layer_original = filter_atoms_by_tag(ase_original_interface, FILM_TAG)
substrate_relaxed = filter_atoms_by_tag(ase_final_interface, SUBSTRATE_TAG)
layer_relaxed = filter_atoms_by_tag(ase_final_interface, FILM_TAG)

original_substrate_energy = calculate_energy(substrate_original, calculator)
original_layer_energy = calculate_energy(layer_original, calculator)
relaxed_substrate_energy = calculate_energy(substrate_relaxed, calculator)
relaxed_layer_energy = calculate_energy(layer_relaxed, calculator)

delta_original = calculate_delta_energy(original_energy, original_substrate_energy, original_layer_energy)
delta_relaxed = calculate_delta_energy(relaxed_energy, relaxed_substrate_energy, relaxed_layer_energy)

area = ase_original_interface.get_volume() / ase_original_interface.cell[2, 2]
n_interface = ase_final_interface.get_global_number_of_atoms()
n_substrate = substrate_relaxed.get_global_number_of_atoms()
n_layer = layer_relaxed.get_global_number_of_atoms()
effective_delta_relaxed = (
                                  relaxed_energy / n_interface
                                  - (relaxed_substrate_energy / n_substrate + relaxed_layer_energy / n_layer)
                          ) / (2 * area)

print(f"Original Substrate energy: {original_substrate_energy:.4f} eV")
print(f"Relaxed Substrate energy:  {relaxed_substrate_energy:.4f} eV")
print(f"Original Layer energy:     {original_layer_energy:.4f} eV")
print(f"Relaxed Layer energy:      {relaxed_layer_energy:.4f} eV")
print("\nDelta between interface energy and sum of component energies")
print(f"Original Delta:            {delta_original:.4f} eV")
print(f"Relaxed Delta:             {delta_relaxed:.4f} eV")
print(f"Original Delta per area:   {delta_original / area:.4f} eV/Ang^2")
print(f"Relaxed Delta per area:    {delta_relaxed / area:.4f} eV/Ang^2")
print(f"Relaxed interface energy:  {relaxed_energy:.4f} eV")
print(
    f"Effective relaxed Delta per area: {effective_delta_relaxed:.4f} eV/Ang^2 ({effective_delta_relaxed / 0.16:.4f} J/m^2)")

/var/folders/wq/kjb0_d9126xd_3j3c13f7n9w0000gn/T/ipykernel_3638/2391716923.py:6: FutureWarning: Please use atoms.calc = calc
  atoms.set_calculator(calc)


Original Substrate energy: -16.1146 eV
Relaxed Substrate energy:  -16.1148 eV
Original Layer energy:     -18.4224 eV
Relaxed Layer energy:      -18.4224 eV

Delta between interface energy and sum of component energies
Original Delta:            0.2057 eV
Relaxed Delta:             0.0008 eV
Original Delta per area:   0.0387 eV/Ang^2
Relaxed Delta per area:    0.0002 eV/Ang^2
Relaxed interface energy:  -34.5364 eV
Effective relaxed Delta per area: 0.7211 eV/Ang^2 (4.5070 J/m^2)


## References

[1] mat3ra-made interface builder: https://github.com/Exabyte-io/made  
[2] MACE-MP-0 foundation model: https://github.com/ACEsuit/mace?tab=readme-ov-file#foundation-models  